# Load Dataset

In [1]:
################################################################################
# Load dataset and split it into training and test set
################################################################################

import pandas as pd
import os
from tabulate import tabulate

dataset_name = "cic-iot"
sample_size = 100000

# Load dateset
df = pd.read_csv(os.getcwd() + f'/data/sample-{sample_size}-2.csv')

# Split dataset according to attack type
normal_df = df[df['label'] == 'BenignTraffic']
attack_df = df[df['label'] != 'BenignTraffic']

# Drop columns
normal_df = normal_df.drop(columns=['label'])
attack_df = attack_df.drop(columns=['label'])

# Split dataset into training and test set
normal_df_train = normal_df.sample(frac=0.8, random_state=42)
normal_df_test = normal_df.drop(normal_df_train.index)
attack_df_train = attack_df.sample(frac=0.8, random_state=42)
attack_df_test = attack_df.drop(attack_df_train.index)

# Print dataset sizes in a table
data = [
    ["Normal", normal_df.shape[0], normal_df_train.shape[0], normal_df_test.shape[0]],
    ["Attack", attack_df.shape[0], attack_df_train.shape[0], attack_df_test.shape[0]]
]
print(tabulate(data, headers=["Atack type", "Total", "Train", "Test"], tablefmt="grid"))

+--------------+---------+---------+--------+
| Atack type   |   Total |   Train |   Test |
+==============+=========+=========+========+
| Normal       |    2348 |    1878 |    470 |
+--------------+---------+---------+--------+
| Attack       |   97652 |   78122 |  19530 |
+--------------+---------+---------+--------+


# Feature Importance

In [2]:
################################################################################
# Generate Feature Importance
################################################################################

import os
import dotenv
import time
import ast
import numpy as np
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Output top 10 important features that can be used to filter an entry as either normal or attack.
Output only in the Python list structure.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```

Example output:
['feature1', 'feature2', 'feature3', ..., 'feature10']
"""

prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatAnthropic(model='claude-3-opus-20240229')
# model_name = "claude-3-opus-20240229"
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name=dataset_name,
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")


normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]

def parse_document(doc):
    try:
        return json.loads(doc)
    except (json.JSONDecodeError, ValueError):
        return ast.literal_eval(doc)

feature_names = normal_df_train.columns.to_list()
doc_length = len(parse_document(normal_documents[0]))
valid_features = feature_names[:doc_length]

normal_entries = {}
for i, feature_name in enumerate(valid_features):
    normal_entries[f"f{i}"] = [parse_document(doc)[i] for doc in normal_documents]

attack_doc_length = len(parse_document(attack_documents[0]))
valid_attack_features = feature_names[:attack_doc_length]

attack_entries = {}
for i, feature_name in enumerate(valid_attack_features):
    attack_entries[f"f{i}"] = [parse_document(doc)[i] for doc in attack_documents]

completions = []
for i in range(10):
    completion = chain.invoke({
        "normal_entries": json.dumps(normal_entries),
        "attack_entries": json.dumps(attack_entries)
    })
    completions.append(completion.content)
    print(completion.content)
    time.sleep(10)

with open(f"results/feature-importance-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write("\n".join(completions))

Looking at the differences between normal and attack entries, I'll analyze the key distinguishing features:

**Key Observations:**

1. **f0**: Normal (40-95 range) vs Attack (0-4 range) - much lower in attacks
2. **f1**: Normal (thousands to millions) vs Attack (54-33625) - significantly lower in attacks
3. **f3**: Normal (67-136 range) vs Attack (all 64) - attacks show constant value
4. **f24**: Normal (481-16363 range) vs Attack (525 or 567) - attacks show only 2 distinct values
5. **f25**: Normal (46-66 range) vs Attack (50 or 54) - attacks show only 2 distinct values
6. **f27**: Normal (87-1220 range) vs Attack (all 50 or 54) - attacks show constant/limited values
7. **f28**: Normal (37-1105 range) vs Attack (all 0) - attacks are always 0
8. **f30**: Normal (mostly near 0, some large values) vs Attack (all 83+ million) - attacks show consistently high values
9. **f31**: Normal (5.5 or 13.5) vs Attack (all 9.5) - different constant values
10. **f33**: Normal (53-1564 range) vs Attac

KeyboardInterrupt: 

# Prediction

In [3]:
################################################################################
# Generate Rules with transposed data
################################################################################

import os
import dotenv
import json
import ast
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import numpy as np
import uuid
# import tiktoken     # https://github.com/openai/tiktoken

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate 5 simple and deterministic rules for top 5 important features to filter an entry as either normal or attack. 
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatAnthropic(model='claude-3-opus-20240229')
# model_name = "claude-3-opus-20240229"
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name=dataset_name,
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]

def parse_document(doc):
    try:
        return json.loads(doc)
    except (json.JSONDecodeError, ValueError):
        return ast.literal_eval(doc)

feature_names = normal_df_train.columns.to_list()
doc_length = len(parse_document(normal_documents[0]))
valid_features = feature_names[:doc_length]

normal_entries = {}
for i, feature_name in enumerate(valid_features):
    normal_entries[feature_name] = [parse_document(doc)[i] for doc in normal_documents]

attack_doc_length = len(parse_document(attack_documents[0]))
valid_attack_features = feature_names[:attack_doc_length]

attack_entries = {}
for i, feature_name in enumerate(valid_attack_features):
    attack_entries[feature_name] = [parse_document(doc)[i] for doc in attack_documents]

# prompt_text = prompt.invoke({
#     "normal_entries": json.dumps(normal_entries),
#     "attack_entries": json.dumps(attack_entries)
# }).text

# print(prompt_text)

completion = chain.invoke({
    "normal_entries": json.dumps(normal_entries),
    "attack_entries": json.dumps(attack_entries)
})

print(completion.content)

id = str(uuid.uuid4())
with open(f"results/llm/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"{completion.content}\n")

# encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
# num_tokens_prompt = len(encoding.encode(prompt.invoke({"normal_entries": json.dumps(normal_entries),"attack_entries": json.dumps(attack_entries)}).text))
# num_tokens_completion = len(encoding.encode(str(completion.content)))

# print(f"Prompt tokens: {num_tokens_prompt}")
# print(f"Completion tokens: {num_tokens_completion}")
# print(f"Total tokens: {num_tokens_prompt + num_tokens_completion}")
# print(f"Percentage of tokens used: {(num_tokens_prompt + num_tokens_completion) / 128000}")

```json
{
  "flow_duration": "ATTACK if flow_duration < 5.0, else NORMAL",
  "IAT": "ATTACK if IAT == 0.0, else NORMAL",
  "Number": "ATTACK if Number == 0.0, else NORMAL",
  "Magnitue": "ATTACK if Magnitue == 0.0, else NORMAL",
  "rst_flag_number": "ATTACK if rst_flag_number > 0.5, else NORMAL"
}
```


In [4]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("attack" if dataset.iloc[i]['flow_duration'] < 1 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['Header_Length'] < 1000 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['Rate'] < 10 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['ack_flag_number'] == 0 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['HTTPS'] == 0 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Max'] == 54 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Protocol Type'] == 6 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Duration'] == 64 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['syn_flag_number'] > 0 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Srate'] < 10 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Drate'] < 10 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['ack_count'] < 1 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Tot sum']< 1000 else "normal")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred, digits=4)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/llm/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}\n")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 19530/19530 [00:00<00:00, 29316.66it/s]


              precision    recall  f1-score   support

      attack     0.9982    0.9625    0.9800     19530
      normal     0.3730    0.9277    0.5320       470

    accuracy                         0.9617     20000
   macro avg     0.6856    0.9451    0.7560     20000
weighted avg     0.9835    0.9617    0.9695     20000

[[18797   733]
 [   34   436]]


# Feedback Loop

In [2]:
################################################################################
# Prompt Template
################################################################################
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

system_message = ("system",
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
"""
)
human_message = ("user",
"""
Analyze the following network data and generate rules for the top 5 important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

# Invoke prompt
# prompt.invoke({"k": 5, "normal_entries": normal_entries, "attack_entries": attack_entries, "msgs": []})

In [4]:
################################################################################
# Define evaluate_rule tool
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = True
operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq, '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value: Annotated[str, "Value"], 
    op: Annotated[str, "Operator"]
) -> bool:
    """Evaluate the rule and return the macro f1-score."""
    try:
        value = float(value)
    except ValueError:
        value
    datasets = {"normal": normal_df_train, "attack": attack_df_train}
    y_pred = []
    y_true = []
    if op in operations:
        for attack_type, dataset in datasets.items():
            test_set_size = dataset.shape[0]
            for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries...", disable=not show_progress):
                y_true.append(attack_type)
                y_pred.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
        c_report = classification_report(y_true, y_pred, digits=4, output_dict=True)
        return c_report['macro avg']['f1-score']
    else:
        raise ValueError(f"Unsupported operator: {op}")

# Invoke tool
# print(evaluate_rule.invoke({"feature_name": "flow_duration", "value": "1", "op": "<"}))

In [5]:
################################################################################
# Initialize LLM
################################################################################

import os
import dotenv
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

model_name = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=model_name, temperature=0.1)
# model_name = "gemini-1.5-pro"
# llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)
# model_name = "claude-3-opus-20240229"
# llm = ChatAnthropic(model=model_name, temperature=0.0)

llm_with_tool = llm.bind_tools([evaluate_rule])

In [6]:
################################################################################
# Set Up Vector Store
################################################################################

import json
import ast
import numpy as np
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

train_set_size = sample_size
n_results = 10
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name=dataset_name,
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=n_results)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=n_results)['documents'][0]

def parse_document(doc):
    try:
        return json.loads(doc)
    except (json.JSONDecodeError, ValueError):
        return ast.literal_eval(doc)

feature_names = normal_df_train.columns.to_list()
doc_length = len(parse_document(normal_documents[0]))
valid_features = feature_names[:doc_length]

normal_entries_dict = {}
for i, feature_name in enumerate(valid_features):
    normal_entries_dict[feature_name] = [parse_document(doc)[i] for doc in normal_documents]

attack_doc_length = len(parse_document(attack_documents[0]))
valid_attack_features = feature_names[:attack_doc_length]

attack_entries_dict = {}
for i, feature_name in enumerate(valid_attack_features):
    attack_entries_dict[feature_name] = [parse_document(doc)[i] for doc in attack_documents]

In [7]:
################################################################################
# Chain with Logging
################################################################################

from langchain_core.messages import HumanMessage

chain = prompt | llm_with_tool

n_repetitions = 5
context_window = 128000
show_progress = False

def get_initial_state():
  n = 0
  k = 5
  mean_f1s = 0
  max_f1s = 0
  n_max = 0
  token_usage = {}
  normal_entries = json.dumps(normal_entries_dict)
  attack_entries = json.dumps(attack_entries_dict)
  msgs = []
  return locals()

def extract_token_usage(ai_msg):
    # langchain-anthropic >= 0.2 uses usage_metadata; older versions use response_metadata["token_usage"]
    if hasattr(ai_msg, "usage_metadata") and ai_msg.usage_metadata:
        meta = ai_msg.usage_metadata
        return {
            "prompt_tokens": meta.get("input_tokens", 0),
            "completion_tokens": meta.get("output_tokens", 0),
            "total_tokens": meta.get("total_tokens", meta.get("input_tokens", 0) + meta.get("output_tokens", 0)),
        }
    token_usage = ai_msg.response_metadata.get("token_usage") or ai_msg.response_metadata.get("usage", {})
    return {
        "prompt_tokens": token_usage.get("prompt_tokens", token_usage.get("input_tokens", 0)),
        "completion_tokens": token_usage.get("completion_tokens", token_usage.get("output_tokens", 0)),
        "total_tokens": token_usage.get("total_tokens", 0),
    }

def extract_rules_from_tool_calls(tool_calls):
    """Extract rule definitions from tool calls.
    
    Handles both ToolCall objects (from ai_msg.tool_calls) and
    raw JSON dictionaries (from additional_kwargs).
    """
    rules = []
    for tool_call in tool_calls:
        try:
            # Handle ToolCall objects (have .args attribute)
            if hasattr(tool_call, 'args') and isinstance(tool_call.args, dict):
                args = tool_call.args
            # Handle raw JSON dictionaries
            elif isinstance(tool_call, dict) and "function" in tool_call:
                args = json.loads(tool_call["function"]["arguments"])
            else:
                continue
            
            rule = {
                "feature_name": str(args.get("feature_name", "")),
                "operator": str(args.get("op", "")),
                "value": str(args.get("value", ""))
            }
            rules.append(rule)
        except (KeyError, AttributeError, TypeError, json.JSONDecodeError) as e:
            print(f"Warning: Could not parse tool call: {e}")
            continue
    return rules

state = get_initial_state()
train_f1_scores = []
refinement_log = []

while state["n"] < n_repetitions:
    ai_msg = chain.invoke(state)
    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        tool_msg = evaluate_rule.invoke(tool_call)
        tool_msgs.append(tool_msg)
    state["mean_f1s"] = sum(float(msg.content) for msg in tool_msgs) / len(tool_msgs)
    
    # Extract selected rules and token usage
    selected_rules = extract_rules_from_tool_calls(ai_msg.tool_calls)
    token_info = extract_token_usage(ai_msg)
    
    # Log this round
    round_entry = {
        "round": state["n"] + 1,
        "macro_f1": state["mean_f1s"],
        "round_prompt_tokens": token_info["prompt_tokens"],
        "round_completion_tokens": token_info["completion_tokens"],
        "round_total_tokens": token_info["total_tokens"],
        "selected_rules": selected_rules
    }
    refinement_log.append(round_entry)
    
    human_msg = HumanMessage(f"The current mean f1-score for the generated rules is {state['mean_f1s']}. "
                             "If this mean f1-score is greater than the previous rounds, keep the better performing "
                             "rules and revise or replace only the underperforming ones (those with a score less than mean). "
                             "Otherwise, revise or replace any rules that have a score less than mean. "
                             f"Based on the feedback, generate exactly {state['k']} rules to filter attack entries and "
                             "make a tool call for each rule, ensuring that a tool call is made for every entry every time.")    
    state["n"] += 1
    state["msgs"].extend([ai_msg, *tool_msgs, human_msg])
    train_f1_scores.append(state["mean_f1s"])
    state["max_f1s"] = state["mean_f1s"] if state["mean_f1s"] > state["max_f1s"] else state["max_f1s"]
    state["n_max"] = state["n"] if state["mean_f1s"] > state["max_f1s"] else state["n_max"]
    state["token_usage"] = extract_token_usage(ai_msg)
    print("Round:", state["n"], "Current mean f1-score:", state["mean_f1s"], "Token usage:", state["token_usage"])

print(train_f1_scores)

# Save refinement log to JSON file
import os
os.makedirs("results/llm", exist_ok=True)

refinement_summary = {
    "dataset": "cic-iot",
    "sample_size": sample_size,
    "seed": 42,
    "model_name": model_name,
    "n_rounds": len(refinement_log),
    "rounds": refinement_log,
    "train_f1_scores": train_f1_scores,
    "max_f1": state["max_f1s"],
    "max_f1_round": state["n_max"]
}

output_file = f"results/llm/policy-refinement-summary-{sample_size}-{model_name}.json"
with open(output_file, "w") as f:
    json.dump(refinement_summary, f, indent=2)

print(f"\nRefinement log saved to: {output_file}")

# Calculate cumulative token usage
for i, round_entry in enumerate(refinement_log):
    if i == 0:
        round_entry["cumulative_prompt_tokens"] = round_entry["round_prompt_tokens"]
        round_entry["cumulative_completion_tokens"] = round_entry["round_completion_tokens"]
        round_entry["cumulative_total_tokens"] = round_entry["round_total_tokens"]
    else:
        prev = refinement_log[i-1]
        round_entry["cumulative_prompt_tokens"] = prev["cumulative_prompt_tokens"] + round_entry["round_prompt_tokens"]
        round_entry["cumulative_completion_tokens"] = prev["cumulative_completion_tokens"] + round_entry["round_completion_tokens"]
        round_entry["cumulative_total_tokens"] = prev["cumulative_total_tokens"] + round_entry["round_total_tokens"]

/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Round: 1 Current mean f1-score: 0.16131464497774822 Token usage: {'prompt_tokens': 5405, 'completion_tokens': 615, 'total_tokens': 6020}


/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Round: 2 Current mean f1-score: 0.3210938844299235 Token usage: {'prompt_tokens': 6318, 'completion_tokens': 582, 'total_tokens': 6900}


/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Round: 3 Current mean f1-score: 0.41046134904163295 Token usage: {'prompt_tokens': 7198, 'completion_tokens': 594, 'total_tokens': 7792}


/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Round: 4 Current mean f1-score: 0.3746224093363003 Token usage: {'prompt_tokens': 8089, 'completion_tokens': 590, 'total_tokens': 8679}
Round: 5 Current mean f1-score: 0.4856666606606999 Token usage: {'prompt_tokens': 8976, 'completion_tokens': 599, 'total_tokens': 9575}
[0.16131464497774822, 0.3210938844299235, 0.41046134904163295, 0.3746224093363003, 0.4856666606606999]

Refinement log saved to: results/llm/policy-refinement-summary-100000-claude-haiku-4-5-20251001.json


In [ ]:
################################################################################
# Evaluate generated rules
################################################################################

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import operator
from statistics import mode

operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq, '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

def evaluate_rules(tool_calls):
    datasets = {"normal": normal_df_test, "attack": attack_df_test}
    y_pred = []
    y_true = []
    for attack_type, dataset in datasets.items():
        test_set_size = dataset.shape[0]
        for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries...", disable=not show_progress):
            predicted_attack_types = []
            for tool_call in tool_calls:
                args = json.loads(tool_call["function"]["arguments"])
                op = args["op"]
                feature_name = args["feature_name"]
                value = args["value"]
                try:
                    value = float(value)
                except ValueError:
                    value
                predicted_attack_types.append("attack" if operations[op](dataset.iloc[i][feature_name], value) else "normal")
            y_true.append(attack_type)
            y_pred.append(mode(predicted_attack_types))
    c_report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    c_matrix = confusion_matrix(y_true, y_pred)
    # print(c_report)
    # print(c_matrix)
    return c_report

# tool_calls = state["msgs"][-7].additional_kwargs["tool_calls"]
# for tool_call in tool_calls:
#     rule = json.loads(tool_call["function"]["arguments"])
#     print("attack if", rule["feature_name"], rule["op"], rule["value"], "else normal")

# evaluate_rules(tool_calls)

# test_f1_scores = []
# for i in range(20, 0, -1):
#     index = -7 * i
#     tool_calls = state["msgs"][index].additional_kwargs["tool_calls"]
#     for tool_call in tool_calls:
#         rule = json.loads(tool_call["function"]["arguments"])
#     test_f1_scores.append(evaluate_rules(tool_calls)['macro avg']['f1-score'])

# print(test_f1_scores)

for i in range(len(state["msgs"])):
    if state["msgs"][i].type != "ai":
        continue
    tool_calls = state["msgs"][i].additional_kwargs["tool_calls"]
    for tool_call in tool_calls:
        rule = json.loads(tool_call["function"]["arguments"])
        print("attack if", rule["feature_name"], rule["op"], rule["value"], "else normal")
    c_report = evaluate_rules(tool_calls)
    print(c_report["macro avg"]["f1-score"])

attack if flow_duration < 1 else normal
attack if Header_Length < 1000 else normal
attack if Duration == 64 else normal
attack if syn_flag_number == 1 else normal
attack if ack_flag_number == 0 else normal
0.8006668027669441
attack if flow_duration < 1 else normal
attack if Header_Length < 1000 else normal
attack if Duration <= 70 else normal
attack if syn_flag_number >= 0.5 else normal
attack if ack_flag_number == 0 else normal
0.9019476275688372
attack if flow_duration < 1 else normal
attack if Header_Length < 1000 else normal
attack if Duration <= 70 else normal
attack if psh_flag_number == 1 else normal
attack if ack_flag_number == 0 else normal
0.9045155019750267
attack if flow_duration < 1 else normal
attack if Header_Length < 1000 else normal
attack if Duration <= 70 else normal
attack if Rate > 50 else normal
attack if ack_flag_number == 0 else normal
0.9426194924959315
attack if flow_duration < 1 else normal
attack if Header_Length < 1000 else normal
attack if Duration <= 70 e

In [8]:
################################################################################
# Evaluate generated rules for efficiency
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate
from statistics import mode
import time
import warnings
import pandas as pd
import os

warnings.filterwarnings("ignore")

sample_size = 100000

# Load dateset
df = pd.read_csv(os.getcwd() + f'/data/sample-{sample_size}-2.csv')

# This dataset does not need categorical encoding as all features are numerical
# except for the label.

# Split dataset according to attack type
normal_df = df[df['label'] == 'BenignTraffic']
attack_df = df[df['label'] != 'BenignTraffic']
normal_df.loc[:, 'label'] = 'normal'
attack_df.loc[:, 'label'] = 'attack'

# Split dataset into training and test set
normal_df_train = normal_df.sample(frac=0.8, random_state=42)
normal_df_test = normal_df.drop(normal_df_train.index)
attack_df_train = attack_df.sample(frac=0.8, random_state=42)
attack_df_test = attack_df.drop(attack_df_train.index)

X_train = pd.concat([normal_df_train, attack_df_train]).drop(columns=['label'])
y_train = pd.concat([normal_df_train, attack_df_train])['label']
X_test = pd.concat([normal_df_test, attack_df_test]).drop(columns=['label'])
y_test = pd.concat([normal_df_test, attack_df_test])['label']

# Create instances of ML models
model_dt = DecisionTreeClassifier()
model_rf = RandomForestClassifier()

# Fit the models to the training data
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

# Predict the labels for the test data
y_true = y_test

elapsed_times_dt = []
elapsed_times_rf = []
elapsed_times_llm = []
y_pred_dt = []
y_pred_rf = []
y_pred_llm = []
for i in range(len(X_test)):
    # Predict using DT
    start = time.time()
    y_pred_dt.append(model_dt.predict([X_test.iloc[i]]))
    end = time.time()
    elapsed_times_dt.append(end - start)

    # Predict using RF
    start = time.time()
    y_pred_rf.append(model_rf.predict([X_test.iloc[i]]))
    end = time.time()
    elapsed_times_rf.append(end - start)
    
    # Predict using LLM
    start = time.time()
    row = X_test.iloc[i]
    conditions = [
        row['flow_duration'] < 1,
        row['Header_Length'] < 1000,
        row['Duration'] <= 70,
        row['Srate'] > 50,
        row['ack_flag_number'] == 0
    ]
    predicted_attack_types = ["attack" if condition else "normal" for condition in conditions]
    y_pred_llm.append(mode(predicted_attack_types))
    end = time.time()
    elapsed_times_llm.append(end - start)

print(f"DT time taken: {sum(elapsed_times_dt)/len(X_test)}")
print(classification_report(y_true, y_pred_dt, digits=4, output_dict=False))
print(confusion_matrix(y_true, y_pred_dt))
print("\n")

print(f"RF time taken: {sum(elapsed_times_rf)/len(X_test)}")
print(classification_report(y_true, y_pred_rf, digits=4, output_dict=False))
print(confusion_matrix(y_true, y_pred_rf))
print("\n")

print(f"LLM time taken: {sum(elapsed_times_llm)/len(X_test)}\n")
print(classification_report(y_true, y_pred_llm, digits=4, output_dict=False))
print(confusion_matrix(y_true, y_pred_llm))

KeyboardInterrupt: 

# Explaination

In [10]:
# write function to use llm to explain the generated rules

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

system_message = ("system",
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
"""
)
human_message = ("user",
"""
Analyze the following network data and generate rules for the top 5 important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

import os
import dotenv
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

model_name = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=model_name, temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)
# model_name = "claude-3-opus-20240229"
# llm = ChatAnthropic(model=model_name, temperature=0.0)

llm_with_tool = llm.bind_tools([evaluate_rule])

# Other

In [11]:
################################################################################
# Generate Rules with Feedback Loop
################################################################################

import os
import dotenv
import json
import ast
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
# import tiktoken     # https://github.com/openai/tiktoken

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You were provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
You were asked to carefully analyze the differences between normal and attack entries by comparing corresponding fields to 
generate 5 simple and deterministic rules for the top 5 important features to filter an entry as either normal or attack.
The rules you generated were evaluated individually against a test set of network data entries, and the following F1-scores were obtained:

F1-scores 1st round:
```
"flow_duration": "if flow_duration < 1 then attack else normal" --> 0.9064
"Header_Length": "if Header_Length <= 1000 then attack else normal" --> 0.8295
"ack_flag_number": "if ack_flag_number == 0 then attack else normal" --> 0.8690
"HTTPS": "if HTTPS == 0 then attack else normal" --> 0.8160
"Max": "if Max == 54.0 then attack else normal" --> 0.6968

Overall F1-score --> 0.9735
```

F1-scores 2nd round:
```
"flow_duration": "if flow_duration < 1 then attack else normal" --> 0.9064
"Header_Length": "if Header_Length <= 100 then attack else normal" --> 0.8265
"ack_flag_number": "if ack_flag_number == 0 then attack else normal" --> 0.8690
"HTTPS": "if HTTPS == 0 then attack else normal" --> 0.8160
"Duration": "if Duration <= 70 then attack else normal" --> 0.3329

Overall F1-score --> 0.9332
```

F1-scores 3rd round:
```
"flow_duration": "if flow_duration < 1 then attack else normal" --> 0.9064
"Header_Length": "if Header_Length <= 100 then attack else normal" --> 0.8265
"ack_flag_number": "if ack_flag_number == 0 then attack else normal" --> 0.8690
"HTTPS": "if HTTPS == 0 then attack else normal" --> 0.8160
"AVG": "if AVG <= 60 then attack else normal" --> 0.9388

Overall F1-score --> 0.9700
```

Based on the feedback provided, drop underperforming rules that has the least f1-score.
Generate new rules to revise the rules to improve the F1-scores.
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", ""),
    ("user", "{}")
])
prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatAnthropic(model='claude-3-opus-20240229')
# model_name = "claude-3-opus-20240229"
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name="cic-iot",
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")


def parse_document(doc):
    try:
        return json.loads(doc)
    except (json.JSONDecodeError, ValueError):
        return ast.literal_eval(doc)


normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]

feature_names = normal_df_train.columns.to_list()
doc_length = len(parse_document(normal_documents[0]))
valid_features = feature_names[:doc_length]

normal_entries = {}
for i, feature_name in enumerate(valid_features):
    normal_entries[feature_name] = [parse_document(doc)[i] for doc in normal_documents]

attack_doc_length = len(parse_document(attack_documents[0]))
valid_attack_features = feature_names[:attack_doc_length]

attack_entries = {}
for i, feature_name in enumerate(valid_attack_features):
    attack_entries[feature_name] = [parse_document(doc)[i] for doc in attack_documents]

# print(prompt.invoke({
#     "normal_entries": json.dumps(normal_entries),
#     "attack_entries": json.dumps(attack_entries)
# }).text)

completion = chain.invoke({
    "normal_entries": json.dumps(normal_entries),
    "attack_entries": json.dumps(attack_entries)
})

print(completion.content)

with open(f"results/llm/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(completion.content)

# encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
# num_tokens_prompt = len(encoding.encode(prompt.invoke({"normal_entries": json.dumps(normal_entries),"attack_entries": json.dumps(attack_entries)}).text))
# num_tokens_completion = len(encoding.encode(str(completion.content)))

# print(f"Prompt tokens: {num_tokens_prompt}")
# print(f"Completion tokens: {num_tokens_completion}")
# print(f"Total tokens: {num_tokens_prompt + num_tokens_completion}")
# print(f"Percentage of tokens used: {(num_tokens_prompt + num_tokens_completion) / 128000}")

```json
{
  "flow_duration": "if flow_duration < 5 then attack else normal",
  "Header_Length": "if Header_Length <= 100 then attack else normal",
  "ack_flag_number": "if ack_flag_number == 0 then attack else normal",
  "AVG": "if AVG > 1000000 then attack else normal",
  "IAT": "if IAT == 0 then attack else normal"
}
```

**Rationale:**
1. **flow_duration**: Increased threshold from <1 to <5 to capture more attack patterns (attacks range 0-4.22 vs normals 40-95)
2. **Header_Length**: Kept at ≤100 (strong F1: 0.8265) - attacks have very small headers
3. **ack_flag_number**: Kept at ==0 (strong F1: 0.8690) - attacks rarely use ACK flags
4. **AVG**: Replaced "Max" rule (F1: 0.6968) with AVG threshold >1000000 (attacks show AVG ~83M vs normals ~0-166M with better separation)
5. **IAT**: Replaced "HTTPS" rule (F1: 0.8160) with IAT==0 (attacks have zero inter-arrival time vs normals 50-1500, providing strong discrimination)


In [12]:
################################################################################
# Generate Rules
################################################################################

import os
import dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate 5 simple and deterministic rules for top 5 important features to filter an entry as either normal or attack. 
Output only in the JSON format with the structure: 
{{'feature1': 'rule', 'feature2': 'rule', ..., 'feature5': 'rule'}}.

Feature Names:
```{feature_names}```

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["feature_names", "normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatAnthropic(model='claude-3-opus-20240229')
# model_name = "claude-3-opus-20240229"
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name="cic-iot",
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]

completion = chain.invoke({
    "feature_names": normal_df_train.columns.to_list(),
    "normal_entries": ",\n".join([f"{doc} --> normal" for doc in normal_documents]),
    "attack_entries": ",\n".join([f"{doc} --> attack" for doc in attack_documents])
    })

print(completion.content)

with open(f"results/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(completion.content)

```json
{
  "flow_duration": "If flow_duration < 5.0, classify as attack; otherwise normal",
  "Duration": "If Duration == 64.0, classify as attack; otherwise normal",
  "Header_Length": "If Header_Length < 200, classify as attack; otherwise normal",
  "Std": "If Std == 0.0, classify as attack; otherwise normal",
  "Covariance": "If Covariance == 0.0, classify as attack; otherwise normal"
}
```


In [13]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        predicted_attack_types = []
        predicted_attack_types.append("attack" if dataset.iloc[i]['flow_duration'] < 1 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['Header_Length'] < 100 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['Duration'] == 64 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['Rate'] < 10 else "normal")
        predicted_attack_types.append("attack" if dataset.iloc[i]['ack_flag_number'] == 0 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['syn_flag_number'] > 0 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Srate'] < 10 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Drate'] < 10 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['ack_count'] < 1 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Tot sum']< 1000 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Max'] < 100 else "normal")
        # predicted_attack_types.append("attack" if dataset.iloc[i]['Protocol Type'] == 6 else "normal")
        y_true.append(attack_type)
        y_pred.append(mode(predicted_attack_types))

c_report = classification_report(y_true, y_pred)
c_matrix = confusion_matrix(y_true, y_pred)

with open(f"results/result-llm-{sample_size}-2.txt", "a") as f:
    f.write(f"Classication Report\n{c_report}\n\nConfusion Matrix\n{c_matrix}")

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|████████████████████████| 19530/19530 [00:00<00:00, 28784.26it/s]


              precision    recall  f1-score   support

      attack       1.00      0.86      0.93     19530
      normal       0.15      1.00      0.26       470

    accuracy                           0.87     20000
   macro avg       0.58      0.93      0.60     20000
weighted avg       0.98      0.87      0.91     20000

[[16892  2638]
 [    0   470]]


In [18]:
################################################################################
# Get a Summary
################################################################################

import dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
# import tiktoken     # https://github.com/openai/tiktoken

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
Given normal and attack network data entries, output human understandable small summary on 
how attack and normal entries can be simply separated.

Feature Names:
```{feature_names}```

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["feature_names", "normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
# llm = ChatGoogleGenerativeAI(model="gemini-1.0-pro")
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name="cic-iot",
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")
retriever = vector_store.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 5, "fetch_k": 5})

normal_documents = retriever.invoke(
    str(normal_df_test.iloc[0].to_list()),
    filter={"$and": [{"source": {"$eq": "cic-iot"}}, {"label": {"$eq": "normal"}}]}
)
attack_documents = retriever.invoke(
    str(attack_df_test.iloc[0].to_list()),
    filter={"$and": [{"source": {"$eq": "cic-iot"}}, {"label": {"$eq": "attack"}}]}
)
completion = chain.invoke({
    "feature_names": normal_df_train.columns.to_list(),
    "normal_entries": ",\n".join([f"{doc.page_content} --> {doc.metadata['label']}" for doc in normal_documents]),
    "attack_entries": ",\n".join([f"{doc.page_content} --> {doc.metadata['label']}" for doc in attack_documents])
    })
print(completion)

content="# Simple Summary: Normal vs Attack Network Traffic\n\n## Key Differences\n\n### **1. Header Length (Most Distinctive)**\n- **Normal**: Very large (140K - 1.3M)\n- **Attack**: Very small (55 - 115)\n- ✅ **Best single separator**\n\n### **2. Flow Duration**\n- **Normal**: Longer flows (0.17 - 3.0 seconds)\n- **Attack**: Much shorter (0.0005 - 4.5 seconds, but with tiny headers)\n\n### **3. Data Rate (Rate/Srate)**\n- **Normal**: High rates (200 - 1600 units)\n- **Attack**: Very low rates (0.47 - 8.7 units)\n\n### **4. Statistical Variance**\n- **Normal**: High variance/std deviation (943 - 2379)\n- **Attack**: Low variance/std deviation (0.07 - 3.15)\n- Indicates normal traffic has more diverse packet sizes; attacks are uniform\n\n### **5. Packet Flags**\n- **Normal**: Mostly PSH flags (1.0), minimal others\n- **Attack**: Mix of FIN, RST, ACK flags with low counts\n- Suggests attacks use different TCP control patterns\n\n### **6. Weight & Magnitude**\n- **Normal**: Weight ~244.6

In [17]:
################################################################################
# Generate Decision Tree
################################################################################

import os
import dotenv
import json
import ast
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
# from langchain_openai import OpenAIEmbeddings
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import numpy as np
import uuid
# import tiktoken     # https://github.com/openai/tiktoken

dotenv.load_dotenv(os.getcwd() + '/../.env')

template = """
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Generate a decision tree with top 5 important features as nodes to filter an entry as either normal or attack. 

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
prompt = PromptTemplate(template=template, input_variables=["normal_entries", "attack_entries"])
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0)
model_name = "claude-haiku-4-5-20251001"
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)
# model_name = "gemini-1.5-pro"
# llm = ChatAnthropic(model='claude-3-opus-20240229')
# model_name = "claude-3-opus-20240229"
chain = prompt | llm
train_set_size = sample_size
embeddings = HuggingFaceEmbeddings()
vector_store = Chroma(
    collection_name=dataset_name,
    embedding_function=embeddings, 
    persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2")

normal_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'normal'})['embeddings']
normal_mean_vector = np.mean(normal_vectors, axis=0).tolist()
normal_documents = vector_store._collection.query(query_embeddings=[normal_mean_vector], n_results=10)['documents'][0]

attack_vectors = vector_store._collection.get(include=['embeddings'], where={'label': 'attack'})['embeddings']
attack_mean_vector = np.mean(attack_vectors, axis=0).tolist()
attack_documents = vector_store._collection.query(query_embeddings=[attack_mean_vector], n_results=10)['documents'][0]

def parse_document(doc):
    try:
        return json.loads(doc)
    except (json.JSONDecodeError, ValueError):
        return ast.literal_eval(doc)

feature_names = normal_df_train.columns.to_list()
doc_length = len(parse_document(normal_documents[0]))
valid_features = feature_names[:doc_length]

normal_entries = {}
for i, feature_name in enumerate(valid_features):
    normal_entries[feature_name] = [parse_document(doc)[i] for doc in normal_documents]

attack_doc_length = len(parse_document(attack_documents[0]))
valid_attack_features = feature_names[:attack_doc_length]

attack_entries = {}
for i, feature_name in enumerate(valid_attack_features):
    attack_entries[feature_name] = [parse_document(doc)[i] for doc in attack_documents]

# prompt_text = prompt.invoke({
#     "normal_entries": json.dumps(normal_entries),
#     "attack_entries": json.dumps(attack_entries)
# }).text

# print(prompt_text)

completion = chain.invoke({
    "normal_entries": json.dumps(normal_entries),
    "attack_entries": json.dumps(attack_entries)
})

print(completion.content)

id = str(uuid.uuid4())
with open(f"results/llm/generated-rules-{sample_size}-llm-{model_name}.txt", "a") as f:
    f.write(f"{id}\n")
    f.write(f"{completion.content}\n")

# encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
# num_tokens_prompt = len(encoding.encode(prompt.invoke({"normal_entries": json.dumps(normal_entries),"attack_entries": json.dumps(attack_entries)}).text))
# num_tokens_completion = len(encoding.encode(str(completion.content)))

# print(f"Prompt tokens: {num_tokens_prompt}")
# print(f"Completion tokens: {num_tokens_completion}")
# print(f"Total tokens: {num_tokens_prompt + num_tokens_completion}")
# print(f"Percentage of tokens used: {(num_tokens_prompt + num_tokens_completion) / 128000}")

# Network Attack Detection Decision Tree

## Top 5 Important Features Analysis

Based on comparing normal vs attack entries, here are the most discriminative features:

| Feature | Normal Range | Attack Range | Importance |
|---------|--------------|--------------|-----------|
| **flow_duration** | 40.7 - 95.3 | 0.0 - 4.2 | ⭐⭐⭐⭐⭐ |
| **Number** | 1,588 - 1,438,113 | 0.0 | ⭐⭐⭐⭐⭐ |
| **IAT** | 53.4 - 1,564.0 | 0.0 | ⭐⭐⭐⭐ |
| **Header_Length** | 3,172 - 4,974,359 | 18.0 - 33,625 | ⭐⭐⭐⭐ |
| **rst_flag_number** | 0.0 | 0.0 - 1.0 | ⭐⭐⭐ |

---

## Decision Tree Structure

```
                    START
                      |
        ┌─────────────┴─────────────┐
        |                           |
   flow_duration                flow_duration
   < 5 seconds?                 >= 5 seconds?
        |                           |
       YES                         NO
        |                           |
    [CHECK: Number]            [CHECK: IAT]
        |                           |
   ┌────┴─

In [16]:
################################################################################
# Evaluate generated rules
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

def classify_traffic(record):
    # Step 1: Check Flow Duration
    if record["flow_duration"] < 1:
        return "attack"
    
    # Step 2: Check Header Length
    if record["Header_Length"] < 200:
        return "attack"
    
    # Step 3: Check Rate
    if record["Rate"] < 10:
        return "attack"
    
    # Step 4: Check IAT
    if record["ack_flag_number"] == 0:
        return "attack"
    
    # Step 5: Check Tot Sum
    if record["Weight"] > 100:
        return "attack"
    
    return "normal"

datasets = {"normal": normal_df_test, "attack": attack_df_test}
y_pred = []
y_true = []
for attack_type, dataset in datasets.items():
    test_set_size = dataset.shape[0]
    for i in tqdm(range(test_set_size), ncols=100, desc=f"Predicting {attack_type} entries..."):
        y_true.append(attack_type)
        y_pred.append(classify_traffic(dataset.iloc[i]))

c_report = classification_report(y_true, y_pred, digits=4)
c_matrix = confusion_matrix(y_true, y_pred)

print(c_report)
print(c_matrix)

Predicting attack entries...: 100%|███████████████████████| 19530/19530 [00:00<00:00, 147445.38it/s]


              precision    recall  f1-score   support

      attack     0.9845    0.9964    0.9904     19530
      normal     0.6979    0.3489    0.4652       470

    accuracy                         0.9811     20000
   macro avg     0.8412    0.6727    0.7278     20000
weighted avg     0.9778    0.9811    0.9781     20000

[[19459    71]
 [  306   164]]
